# Clasificación de Letras Escritas a Mano con Redes Convolucionales (CNNs)

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Asignatura Ministerial:** Procesamiento Digital de Imágenes
* **Nombre de Trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

El objetivo de esta sesión es diseñar, entrenar y evaluar una **Red Neuronal Convolucional (CNN)** para clasificar automáticamente imágenes de letras escritas a mano (dataset EMNIST). Estudiaremos la adición de filtros convolucionales 2D y capas de reducción espacial, contrastando directamente la superioridad de esta arquitectura espacial frente al modelo de Perceptrón Multicapa (MLP) analizado en el cuaderno anterior.

## Resultados de aprendizaje

Al final de este notebook van a poder:
1. Explicar el propósito y ventajas de las capas convolucionales 2D en Procesamiento Digital de Imágenes.
2. Diseñar una arquitectura convolucional secuencial uniendo capas `Conv2D`, `MaxPooling2D`, `Flatten` y `Dense`.
3. Entrenar y graficar las curvas de precisión y error de una CNN.
4. Comparar empíricamente y teóricamente el desempeño de un Perceptrón Multicapa (MLP) contra una CNN sobre el mismo conjunto de datos.

## Terminología clave (Microglosario)

*   **Red Convolucional (CNN):** Modelo de aprendizaje profundo diseñado para procesar datos bidimensionales preservando sus relaciones espaciales y de vecindad.
*   **Capa Convolucional (`Conv2D`):** Capa que desplaza un conjunto de filtros (*kernels*) sobre la imagen para extraer mapas de características locales como bordes o contornos.
- **kernels** es una pequeña matriz de números que recorre la imagen para detectar patrones. Una capa suele tener muchos kernels. Cada uno produce su propio mapa de características.
*   **Capa de Reducción (`MaxPooling2D`):** Operación que reduce el ancho y alto de los mapas de características reteniendo únicamente la activación máxima en pequeñas regiones.
*   **Invarianza Espacial:** Capacidad del modelo de reconocer un patrón o contorno sin importar su posición o ligera traslación dentro de la imagen.


## 1. Configuración de Librerías y Herramientas

Carguemos las herramientas esenciales de TensorFlow, visualización y análisis cuantitativo de datos.

In [1]:
# Comentario general: este bloque prepara el entorno instalando librerias o dependencias necesarias; sin esas librerias el resto del notebook no podria importar modelos, cargar datos o construir la interfaz.
print("✦ Instalando dependencias en el sistema...")  # Muestra informacion en consola para verificar el avance o los resultados.
!pip install seaborn -q  # Ejecuta un comando especial del notebook o del sistema.
print("✓ Librerías instaladas con éxito.\n")  # Muestra informacion en consola para verificar el avance o los resultados.

import tensorflow as tf  # Importa tensorflow para usar sus funciones en el notebook.
import tensorflow_datasets as tfds  # Importa tensorflow_datasets para usar sus funciones en el notebook.
import matplotlib.pyplot as plt  # Importa matplotlib para usar sus funciones en el notebook.
import numpy as np  # Importa numpy para usar sus funciones en el notebook.
import math  # Importa math para usar sus funciones en el notebook.
import os  # Importa os para usar sus funciones en el notebook.
import seaborn as sns  # Importa seaborn para usar sus funciones en el notebook.
from sklearn.metrics import confusion_matrix  # Importa componentes especificos desde sklearn.metrics.

print("✓ Entorno de Deep Learning listo para operar.")  # Muestra informacion en consola para verificar el avance o los resultados.

✦ Instalando dependencias en el sistema...



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


✓ Librerías instaladas con éxito.

✓ Entorno de Deep Learning listo para operar.


## 2. Carga y Preprocesamiento de EMNIST

Para realizar una comparación justa y directa contra el modelo anterior, utilizaremos el mismo dataset de letras escritas a mano (EMNIST/Letters) bajo idénticos parámetros de normalización y estructuración de etiquetas.

In [2]:
print("✦ Descargando y cargando EMNIST/Letters en memoria...")  
datos, metadatos = tfds.load('emnist/letters', as_supervised=True, with_info=True)  
print("✓ Dataset cargado correctamente.\n")  

# Generamos de forma explícita el vector de etiquetas
nombres_clases = []  
for i in range(1, 27):  
    letra = chr(i + ord('a') - 1)  
    nombres_clases.append(letra)  

print("✓ Clases listas:", nombres_clases)  

✦ Descargando y cargando EMNIST/Letters en memoria...
✓ Dataset cargado correctamente.

✓ Clases listas: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [3]:
# Este bloque separa datos de entrenamiento, validacion o prueba.
def preprocesar_imagen(imagen, etiqueta):  
    # Convertimos a flotante y normalizamos al rango [0, 1]
    imagen_normalizada = tf.cast(imagen, tf.float32) / 255.0  # Convierte el tipo de dato para operar numericamente con TensorFlow.
    # Desplazamos la etiqueta para iniciar en 0
    etiqueta_ajustada = etiqueta - 1  
    return imagen_normalizada, etiqueta_ajustada  

# Preparamos los conjuntos de datos en caché para máxima velocidad
datos_entrenamiento = datos['train'].map(preprocesar_imagen).cache()  
datos_pruebas = datos['test'].map(preprocesar_imagen).cache()  

print("✓ Datos preprocesados con éxito.")  

✓ Datos preprocesados con éxito.


## 3. Preparación de Lotes para el Entrenamiento

Agruparemos las muestras en lotes de tamaño 32 y las barajaremos para asegurar un gradiente de optimización equilibrado y libre de sesgos secuenciales.

In [4]:
# Configura el pipeline de entrada para entrenamiento y evaluación.
# El dataset de entrenamiento se mezcla, divide en lotes y se hace repetir indefinidamente;
# el de pruebas solo se divide en lotes (el orden no afecta la evaluación).

TAMANO_LOTE = 32

total_muestras_entrenamiento = metadatos.splits['train'].num_examples

# buffer_size = total de muestras garantiza un shuffle completo (aleatorio uniforme).
# Un buffer más pequeño daría mezcla parcial y podría introducir sesgo por orden original.
datos_entrenamiento_lotes = datos_entrenamiento.shuffle(total_muestras_entrenamiento)
datos_entrenamiento_lotes = datos_entrenamiento_lotes.batch(TAMANO_LOTE)
# repeat() hace que el dataset se reanude automáticamente al agotarse.
# Es necesario cuando se entrena con steps_per_epoch, ya que Keras consume
# más pasos que ejemplos hay en un único ciclo del dataset.
datos_entrenamiento_lotes = datos_entrenamiento_lotes.repeat()

# Sin shuffle ni repeat: la evaluación recorre el conjunto de pruebas una sola vez en orden.
datos_pruebas_lotes = datos_pruebas.batch(TAMANO_LOTE)

print(f"✓ Lotes de {TAMANO_LOTE} configurados correctamente para la CNN.")


✓ Lotes de 32 configurados correctamente para la CNN.


## 4. Diseño y Construcción de la Red Neuronal Convolucional (CNN)

A diferencia del Perceptrón Multicapa, la **CNN** recibe y procesa la imagen conservando su formato original 2D de $28 \times 28 \times 1$. Aplicaremos filtros convolucionales alternados con capas de Max-Pooling para extraer y resumir rasgos geométricos abstractos.

In [5]:
# CNN para clasificar letras manuscritas (28×28 px, escala de grises).
# Dos bloques Conv→Pool extraen características espaciales en escala creciente de abstracción;
# las capas Dense combinan esas características y producen una probabilidad por clase.

modelo_cnn = tf.keras.Sequential([
    # input_shape=(28, 28, 1): alto, ancho, canales — 1 canal indica escala de grises (no RGB).
    tf.keras.layers.Conv2D(16, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # 32 filtros (el doble que la capa anterior): a mayor profundidad, mayor capacidad
    # necesaria para representar patrones más complejos (combinaciones de bordes y curvas).
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),

    # softmax convierte los scores en una distribución de probabilidad que suma 1.
    # len(nombres_clases) hace la arquitectura independiente del número de clases.
    tf.keras.layers.Dense(len(nombres_clases), activation='softmax')
])

modelo_cnn.compile(
    optimizer='adam',
    # sparse_categorical_crossentropy acepta etiquetas como enteros directamente,
    # sin requerir conversión previa a one-hot encoding.
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✓ Arquitectura de la CNN diseñada y compilada.")
modelo_cnn.summary()


c:\Proyectos\rodriguez-carmen-pdi-1c-2026\venv312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


✓ Arquitectura de la CNN diseñada y compilada.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 800)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 26)             │         1,690 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57,754 (225.60 KB)

 Trainable params: 57,754 (225.60 KB)

 Non-trainable params: 0 (0.00 B)

Esta salida no muestra todavía la precisión del modelo: es el resumen de la arquitectura de la CNN.

La entrada se infiere como una imagen de 28 × 28 × 1 píxeles, una letra en escala de grises.

Conv2D → (26, 26, 16): aplica 16 filtros de 3 × 3. Cada filtro genera un mapa que detecta características como bordes o trazos. Tiene 160 parámetros entrenables.

MaxPooling2D → (13, 13, 16): reduce cada mapa a la mitad, conservando las características principales. No aprende parámetros.

Segunda Conv2D → (11, 11, 32): aplica 32 filtros para reconocer patrones más complejos. Tiene 4.640 parámetros.

Segundo MaxPooling2D → (5, 5, 32): vuelve a reducir la información espacial.
Flatten → 800: transforma los 5 × 5 × 32 = 800 valores en un vector.

Dense → 64: combina las características para decidir qué letra representan. Esta capa concentra 51.264 parámetros.

Dense → 26: genera una puntuación o probabilidad para cada una de las 26 letras.
None representa el tamaño variable del lote: el modelo puede procesar una o varias imágenes simultáneamente.

En total hay 57.754 parámetros, todos entrenables. La mayor parte está en la capa Dense(64), no en las convoluciones.

## Consigna de Lectura e Interpretación

Revisen el bloque de `summary()`. Observen la transición de las dimensiones espaciales de salida a medida que la imagen atraviesa la primera y segunda capa de `Conv2D` y `MaxPooling2D`. ¿Cómo cambia el tamaño del tensor?

**Respuesta orientativa:** En una CNN las dimensiones espaciales suelen disminuir despus de convoluciones y pooling, mientras aumenta o cambia la cantidad de canales. Esto significa que la red comprime la imagen y conserva rasgos utiles para la clasificacion.

## 5. Entrenamiento de la CNN

Procederemos al entrenamiento del modelo por un total de 15 épocas completas, análogo al entrenamiento del MLP anterior.

In [6]:
# Entrena la CNN calculando explícitamente cuántos lotes conforman una época,
# necesario porque el dataset de entrenamiento usa .repeat() y no tiene fin natural.

EPOCAS = 15
# ceil garantiza que todos los ejemplos se vean aunque no sean divisibles exactamente
# por TAMANO_LOTE; sin steps_per_epoch Keras no sabría cuándo termina cada época.
pasos_por_epoca = math.ceil(total_muestras_entrenamiento / TAMANO_LOTE)

print(f"✦ Iniciando entrenamiento de la CNN por {EPOCAS} épocas...")
print("  (Observen la precisión alcanzada desde las primeras épocas)\n")

historial = modelo_cnn.fit(
    datos_entrenamiento_lotes,
    epochs=EPOCAS,
    steps_per_epoch=pasos_por_epoca
)

print("\n✓ Entrenamiento finalizado con éxito.")


✦ Iniciando entrenamiento de la CNN por 15 épocas...
  (Observen la precisión alcanzada desde las primeras épocas)

Epoch 1/15
2775/2775 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.8181 - loss: 0.5890
Epoch 2/15
2775/2775 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.8996 - loss: 0.3080
Epoch 3/15
2775/2775 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9161 - loss: 0.2538
Epoch 4/15
2775/2775 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9259 - loss: 0.2204
Epoch 5/15
2775/2775 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9331 - loss: 0.1957
Epoch 6/15
2775/2775 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9379 - loss: 0.1777
Epoch 7/15
2775/2775 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9427 - loss: 0.1625
Epoch 8/15
2775/2775 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9465 - loss: 0.1496
Epoch 9/15
2775/2775 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9499 - loss: 0.1389
Epoch 10/15
2775/2775 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9517 - loss: 0.1299
Epo

## 6. Evaluación de la CNN en Datos de Prueba

Validaremos la exactitud final alcanzada en el conjunto de pruebas que el modelo no procesó durante su optimización.

In [7]:
# Evalúa la CNN sobre el conjunto de prueba para medir su capacidad de generalización
# (rendimiento sobre datos que el modelo no vio durante el entrenamiento).

print("✦ Evaluando la CNN en datos no vistos...")

# evaluate devuelve las métricas en el mismo orden en que se definieron en compile():
# primero la pérdida, luego accuracy. El orden del desempaquetado debe respetarlo.
perdida_prueba, precision_prueba = modelo_cnn.evaluate(datos_pruebas_lotes, verbose=0)

print(f"\nResultados en Datos de Prueba (CNN):")
print(f"  • Pérdida (Loss) calculada: {perdida_prueba:.4f}")
print(f"  • Precisión (Accuracy) calculada: {precision_prueba:.4f} ({precision_prueba * 100:.2f}%)")


✦ Evaluando la CNN en datos no vistos...

Resultados en Datos de Prueba (CNN):
  • Pérdida (Loss) calculada: 0.3146
  • Precisión (Accuracy) calculada: 0.9146 (91.46%)


## 7. Contraste y Comparación Analítica: MLP vs. CNN

A continuación, realizaremos una comparación de rendimiento entre ambos clasificadores basándonos en sus arquitecturas matemáticas.

### Cuadro Comparativo Teórico-Empírico

| Métrica / Aspecto | Perceptrón Multicapa (MLP - Cuaderno 02) | Red Convolucional (CNN - Cuaderno 03) |
| :--- | :--- | :--- |
| **Tratamiento Espacial** | Aplanado unidimensional (destruye topología) | Preserva la estructura 2D bidimensional |
| **Extracción de Patrones** | Manual implícita global | Convolución local jerárquica con filtros |
| **Invarianza a Traslaciones** | Extremadamente baja (sensible al desfase) | Muy alta (robusta ante ligeros desplazamientos) |
| **Precisión Promedio (EMNIST)**| ~80% - 84% | **~90% - 94%** |
| **Parámetros del summary()** | Elevados en primera capa | Distribución equilibrada gracias al Max-Pooling |

Tratamiento espacial: el MLP aplana la imagen y pierde la representación explícita de filas, columnas y vecindades. La CNN conserva esa estructura bidimensional.

Extracción de patrones: el MLP conecta globalmente los píxeles; la CNN aprende filtros locales para detectar bordes, curvas, texturas y formas.

Desplazamientos: el MLP es sensible a que una letra aparezca ligeramente corrida. La convolución y el pooling hacen a la CNN más robusta ante esos cambios.

Precisión: según la tabla, el MLP alcanza aproximadamente 80–84 %, mientras que la CNN llega a 90–94 %.

Parámetros: el MLP necesita muchas conexiones densas desde el comienzo. La CNN comparte los mismos filtros en toda la imagen y el Max-Pooling reduce el tamaño de los mapas, disminuyendo los parámetros de las capas posteriores.

Conviene matizar que el Max-Pooling no “equilibra” directamente los parámetros: reduce las dimensiones espaciales y, por ello, la cantidad de conexiones necesarias después.

## Consigna de Lectura e Interpretación

Analicen la tabla anterior y confronten los resultados empíricos que obtuvieron en el laboratorio. ¿A qué se debe el gran incremento en precisión de la CNN por sobre el MLP? ¿De qué forma afecta el hecho de aplanar la imagen en un vector unidimensional al procesamiento de contornos complejosi

La ventaja de una CNN es que conserva la estructura espacial de la imagen. Mientras una red densa trata los pixeles como una lista, la CNN aprende patrones locales: bordes, esquinas, texturas y combinaciones de formas.

El incremento de precisión de la CNN se debe principalmente a que conserva y aprovecha la estructura espacial de la imagen. Sus filtros analizan regiones pequeñas y aprenden progresivamente características como bordes, esquinas, trazos y combinaciones de formas. Además, los mismos filtros se aplican en distintas posiciones, por lo que la red puede reconocer una letra aunque esté ligeramente desplazada.

Al aplanar la imagen, el MLP transforma la matriz bidimensional en una lista. Los píxeles no desaparecen, pero la red deja de disponer de una representación explícita de cuáles son vecinos. Por eso debe aprender mediante conexiones densas relaciones espaciales que la CNN incorpora directamente en su arquitectura. Esto dificulta el reconocimiento de contornos complejos y vuelve al MLP más sensible a cambios de posición, escala o pequeñas deformaciones.

En consecuencia, la CNN consigue mejores resultados porque está diseñada específicamente para aprender patrones visuales, mientras que el MLP trata la imagen como un conjunto general de valores.

## Cierre de Laboratorio

Han comprobado empíricamente la enorme diferencia en rendimiento y robustez espacial que aporta una Red Convolucional (CNN) en Procesamiento Digital de Imágenes en comparación con un Perceptrón clásico.

En la siguiente sesión ingresaremos al núcleo de estas capas de convolución y pooling para visualizar de manera exacta cómo reaccionan las neuronas e inspeccionar el contenido de sus filtros.